# Phase 1 tests
1. **Unit tests** of the transformation and quality functions on the small files in `data/`:
   `sample_rides.csv` (clean, 152 rows) and `corrupted_rides.csv` (same rows with deliberate faults).
2. **End-to-end check** of the full tables written by the Databricks Job.

Run all cells top to bottom. Every test prints ✓ or stops with an error.

In [ ]:
import os
from pyspark.sql import functions as F
from src import config
from src import etl_pipeline as etl
from src import gold_analytics as gold
from src import quality_checks as qc

config.configure_adls(spark)
DATA = "file:" + os.path.join(os.path.dirname(os.getcwd()), "data")   # repo's data/ folder

def check(name, condition):
    assert condition, f"FAILED: {name}"
    print("✓", name)

## Bronze on clean sample data

In [ ]:
bronze = etl.to_bronze(etl.read_raw(spark, f"{DATA}/sample_rides.csv"), "test")
check("all 152 rows read", bronze.count() == 152)
check("quotes stripped from IDs", bronze.filter(F.col("booking_id").contains('"') | F.col("customer_id").contains('"')).count() == 0)
check("text null is a real null", bronze.filter(F.col("booking_status") == "No Driver Found").filter(F.col("booking_value").isNotNull()).count() == 0)
check("metadata columns added", {"ingestion_timestamp", "source_file", "batch_id"} <= set(bronze.columns))
check("dates parsed", bronze.filter(F.col("booking_date").isNull()).count() == 0)

## Silver on clean sample data

In [ ]:
silver, quarantine = etl.to_silver(bronze, "test")
check("bronze = silver + quarantine", silver.count() + quarantine.count() == 152)
check("no duplicate Booking IDs in Silver", silver.count() == silver.select("booking_id").distinct().count())
check("the 1 duplicate Booking ID in the sample is quarantined", quarantine.filter(F.col("reject_reason") == "duplicate_booking_id").count() == 1)
check("booking_timestamp built", silver.filter(F.col("booking_timestamp").isNull()).count() == 0)
row = silver.filter(F.col("booking_status") == "Cancelled by Driver").first()
check("derived cancellation columns", row["is_cancelled"] and row["cancelled_by"] == "driver" and row["cancellation_reason"] == row["driver_cancel_reason"])
check("derived hour and day", row["booking_hour"] == row["booking_timestamp"].hour and row["day_of_week"] == row["booking_date"].strftime("%A"))
display(silver.limit(5))

## Quality checks pass on clean data
(Timeliness and volume are skipped here: the sample only covers some days.)

In [ ]:
results = {c["rule"]: c for c in qc.bronze_checks(spark, bronze) + qc.silver_checks(bronze, silver, quarantine)}
for rule, c in results.items():
    if rule not in ("timeliness_missing_days", "volume_vs_previous_load"):
        check(f"{rule} passes", c["status"] == "PASS")

## Gold on sample data

In [ ]:
daily = gold.daily_bookings(silver)
check("daily totals reconcile with Silver", daily.agg(F.sum("total_bookings")).first()[0] == silver.count())
for name, build in gold.BUILDERS.items():
    check(f"gold table {name} has rows", build(silver).count() > 0)

## Deliberately corrupted data: rows are quarantined and checks fail

In [ ]:
bad_bronze = etl.to_bronze(etl.read_raw(spark, f"{DATA}/corrupted_rides.csv"), "test")
bad_silver, bad_quarantine = etl.to_silver(bad_bronze, "test")
reasons = {r[0] for r in bad_quarantine.select(F.explode(F.split("reject_reason", ","))).collect()}
for expected in ["negative_booking_value", "driver_rating_outside_1_to_5", "completed_without_payment_method",
                 "cancelled_by_driver_flag_does_not_match_status", "missing_customer_id",
                 "not_a_number_ride_distance", "duplicate_booking_id"]:
    check(f"quarantined: {expected}", expected in reasons)
display(bad_quarantine.select("booking_id", "booking_status", "reject_reason"))

bad = {c["rule"]: c for c in qc.bronze_checks(spark, bad_bronze)}
for rule in ["completeness_key_columns", "completed_rides_missing_payment_method", "uniqueness_booking_id_bronze",
             "validity_ratings_1_to_5", "validity_non_negative_values", "consistency_flag_matches_status"]:
    check(f"{rule} fails on corrupted data", bad[rule]["status"] == "FAIL")
print(bad["validity_non_negative_values"])   # structured result with sample failing records

## End-to-end: full tables written by the Databricks Job
Run this after the Job has completed (README step 5). 📸 Screenshot the results table.

In [ ]:
out = qc.run_quality_checks(spark, ["bronze", "silver", "gold"])
print("Overall:", out["status"], "| hard failures:", out["hard_failed"])
display(spark.createDataFrame([{k: str(v) for k, v in c.items()} for c in out["checks"]]))
for layer, path in [("Bronze", config.BRONZE_PATH), ("Silver", config.SILVER_PATH), ("Quarantine", config.QUARANTINE_PATH)]:
    print(layer, spark.read.format("delta").load(path).count())